Nakdimon Dataset
This notebook make Hebrew fluers dataset with nikud

In [ ]:
# conda create -n nakdimon_env python=3.11 -y
# conda activate nakdimon_env
# pip install nakdimon
# conda install -n nakdimon_env ipykernel --update-deps --force-reinstall

#conda create -n nakdimon_env python=3.11 cudatoolkit=11.8.0 cudnn=8.9.2.26 -c conda-forge -y
# conda activate nakdimon_env
# pip install tensorflow==2.15.0 nakdimon ipykernel
# pip install tourch
# pip install torch
# pip install torchaudio
# pip install phonemizer
# pip install datasets
# pip install ipywidgets

In [2]:
import os
import json
import torch
import torchaudio
import nakdimon
import time
from data.tokenizer import AudioTokenizer, TextTokenizer
import re
from datasets import concatenate_datasets, load_dataset
from tqdm import tqdm
import subprocess
import random

In [18]:
# Downloading and Loading dataset 
print("Downloading and Loading the Fleurs dataset...")

try:
    # Train dataset
    print("Train dataset...")
    dataset_train_orig = load_dataset(
        "google/fleurs", 
        "he_il", 
        split="train", 
        trust_remote_code=True
    )
    
    print("\n--- Success! ---")
    print(f"Train Dataset loaded. Number of samples: {len(dataset_train_orig)}")
    
    # Check one sample to make sure audio and text are linked
    sample = dataset_train_orig[1388]
    print(f"Text: {sample['transcription']}")
    print(f"Audio array shape: {sample['audio']['array'].shape}")

except Exception as e:
    print(f"Error loading: {e}")
    print("If it fails, we will try to point it directly to the download folder.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Train dataset...


Using the latest cached version of the dataset since google/fleurs couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'he_il' at /home/sukiennik/.cache/huggingface/datasets/google___fleurs/he_il/2.0.0/80cb68d1b4d319aefbd8ea302274d3950d95f6242f0742c1452d1545c80a2d5f (last modified on Tue Mar  3 11:00:04 2026).



--- Success! ---
Train Dataset loaded. Number of samples: 3242
Text: המחאה החלה בערך ב-11:00 זמן מקומי utcּ+1 בווייטהול מול הכניסה לרחוב דאונינג השמורה על ידי שוטרים המשכן הרשמי של ראש הממשלה
Audio array shape: (211200,)


In [26]:
    sample = dataset_train_orig[1065]
    print(f"Text: {sample['transcription']}")
    print(f"Audio array shape: {sample['audio']['array'].shape}")

Text: המחאה החלה בערך ב-11:00 זמן מקומי utcּ+1 בווייטהול מול הכניסה לרחוב דאונינג השמורה על ידי שוטרים המשכן הרשמי של ראש הממשלה
Audio array shape: (202560,)


In [ ]:
try:
    # Validation dataset
    print("Validation dataset...")
    dataset_val = load_dataset(
        "google/fleurs", 
        "he_il", 
        split="validation", 
        trust_remote_code=True
    )
    
    print("\n--- Success! ---")
    print(f"Validation Dataset loaded. Number of samples: {len(dataset_val)}")
    
    # Check one sample to make sure audio and text are linked
    sample = dataset_val[0]
    print(f"Text: {sample['transcription']}")
    print(f"Audio array shape: {sample['audio']['array'].shape}")

except Exception as e:
    print(f"Error loading: {e}")
    print("If it fails, we will try to point it directly to the download folder.")

In [ ]:
try:
    # Test dataset
    print("Test dataset...")
    dataset_test_orig = load_dataset(
        "google/fleurs", 
        "he_il", 
        split="test", 
        trust_remote_code=True
    )
    
    print("\n--- Success! ---")
    print(f"Test Dataset loaded. Number of samples: {len(dataset_test_orig)}")
    
    # Check one sample to make sure audio and text are linked
    sample = dataset_test_orig[0]
    print(f"Text: {sample['transcription']}")
    print(f"Audio array shape: {sample['audio']['array'].shape}")

except Exception as e:
    print(f"Error loading: {e}")
    print("If it fails, we will try to point it directly to the download folder.")

In [ ]:
# Re-balancing the dataset to maximize training data

# Split the Test into: 700 for Train, 91 for final Test
test_for_train = dataset_test_orig.select(range(700))
dataset_test = dataset_test_orig.select(range(700, len(dataset_test_orig)))

# Combine original Train with the extra Test samples
dataset_train = concatenate_datasets([dataset_train_orig, test_for_train])

print(f"New Train size: {len(dataset_train)}")
print(f"Final Test size: {len(dataset_test)}")

In [2]:
import nakdimon
print(dir(nakdimon))

['MAIN_MODEL', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'argparse', 'config', 'diacritize', 'diacritize_main', 'do_metrics', 'do_predict', 'do_run_test', 'do_server', 'do_train', 'logging', 'main', 'os', 'sys']


In [2]:
import os

# הדרך הבטוחה: הגדרת הנתיב רק עבור הסקריפט הזה
# זה לא משנה כלום במערכת ההפעלה שלך באופן קבוע
if 'CONDA_PREFIX' in os.environ:
    conda_lib_path = os.path.join(os.environ['CONDA_PREFIX'], 'lib')
    os.environ['LD_LIBRARY_PATH'] = conda_lib_path + ":" + os.environ.get('LD_LIBRARY_PATH', '')

import tensorflow as tf
import nakdimon

# עכשיו נבדוק
print("GPU list:", tf.config.list_physical_devices('GPU'))

2026-03-07 20:40:56.292114: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 20:40:56.414298: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-07 20:40:56.414367: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-07 20:40:56.421513: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-07 20:40:56.448214: I tensorflow/core/platform/cpu_feature_guar

GPU list: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-03-07 20:40:58.439864: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-07 20:40:58.440146: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-07 20:40:58.440167: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [3]:
import tensorflow as tf
print("Devices detected:", tf.config.list_physical_devices())
# You want to see 'GPU' in the output list

Devices detected: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
import tensorflow as tf

# Check if TensorFlow was built with CUDA (GPU support)
print("Built with CUDA:", tf.test.is_built_with_cuda())

# List all available physical devices
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ Success! Found {len(gpus)} GPU(s):")
    for gpu in gpus:
        print(f"  - {gpu}")
else:
    print("❌ GPU still not detected. Running on CPU.")

Built with CUDA: True
✅ Success! Found 1 GPU(s):
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [4]:
import nakdimon

# The main function in the package seems to be 'diacritize'
text = "שלום עולם"

# Run the diacritization process
# Note: This usually takes a few seconds on first run to load models
result = nakdimon.diacritize(text)

print(result)

2026-03-07 11:18:34.840895: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 11:18:34.894563: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-07 11:18:35.147910: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-07 11:18:35.147998: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-07 11:18:35.194420: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

1/1 [==============================] - 6s 6s/step
שָׁלוֹם עוֹלַם


In [2]:
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")

In [6]:
# Test cases to check how the tokenizer handles diacritics
test_sentences = [
    "דגש",          # No diacritics
    "דָּגָשׁ",       # With Dagesh and Kamatz
    "ספר",          # Could be Sefer or Sfar
    "סֵפֶר",         # Sefer (Book)
    "סְפָר"          # Sfar (Border)
]

print("--- Phoneme Tokenization Test ---")
for text in test_sentences:
    # Running your specific tokenizer
    phonemes = text_tokenizer_he(text)[0]
    phonemes_str = " ".join(phonemes)
    print(f"Text: {text:10} | Phonemes: {phonemes_str}")

--- Phoneme Tokenization Test ---
Text: דגש        | Phonemes: d ɡ ʃ
Text: דָּגָשׁ    | Phonemes: d a ɡ a ʃ
Text: ספר        | Phonemes: s e f e ʁ
Text: סֵפֶר      | Phonemes: s e f e ʁ
Text: סְפָר      | Phonemes: s f a ʁ


In [7]:
# Final test for B/V, K/Kh, P/F distinction
test_pairs = [
    ("בַּיִת", "בּ"), # Bayit (B)
    ("גַּב", "ב"),    # Gav (V)
    ("כַּלְבָּה", "כּ"), # Kalba (K)
    ("מֶלֶךְ", "ך"),   # Melekh (Kh)
    ("פִּיל", "פּ"),   # Pil (P)
    ("קוף", "פ")     # Kof (F)
]

print("--- Begadkephat Phoneme Test ---")
for word, char in test_pairs:
    try:
        phonemes = text_tokenizer_he(word)[0]
        print(f"Word: {word:10} | Phonemes: {' '.join(phonemes)}")
    except Exception as e:
        print(f"Error: {e}")

--- Begadkephat Phoneme Test ---
Word: בַּיִת     | Phonemes: b a i t
Word: גַּב       | Phonemes: ɡ i m e l _ ( en ) h iː b ɹ uː d a ɡ ɛ ʃ ( he ) _ a _ v e t
Word: כַּלְבָּה  | Phonemes: k a l b a ʔ
Word: מֶלֶךְ     | Phonemes: m e l e χ
Word: פִּיל      | Phonemes: p i j l
Word: קוף        | Phonemes: k v f


In [33]:
# --- Configuration ---
manifest_dir = "./voicecraft_data_nakdimon/manifest"
splits = {
    "train": "train_manifest_filtered.jsonl",
    "val": "val_manifest_filtered.jsonl",
    "test": "test_manifest_filtered.jsonl"
}

# --- Helper Functions ---
def final_cleanup(text):
    """ Cleans text artifacts before diacritization """
    if not text: return ""
    return " ".join(text.replace('\\', '').replace('"', "'").split())

def fix_for_phonemes(text):
    """ 
    Normalizes Hebrew text specifically for the phonemizer to prevent 
    English descriptions (hallucinations) of letters and vowels.
    """
    if not text: return ""
    # 1. Global fix: Convert Holam Haser to Holam Male for all characters
    # Matches any char (except Vav) followed by Holam Haser and adds a Vav
    text = re.sub(r'([^ו])ֹ', r'\1וֹ', text)
    
    # 2. Specific fix for "Lo" (לֹא) which often fails
    text = text.replace("לֹא", "לוֹא")
    
    # 3. Clean up: Remove Dagesh and Shin/Sin dots that cause "spelling out" errors
    text = text.replace('ּ', '').replace('ׁ', '').replace('ׂ', '')
    
    return text

# Ensure the 'tests' directory exists (Nakdimon CLI requirement)
if not os.path.exists('tests'):
    os.makedirs('tests')

for split_name, filename in splits.items():
    input_manifest = os.path.join(manifest_dir, filename)
    output_manifest = os.path.join(manifest_dir, f"{split_name}_manifest_nikud.jsonl")
    
    temp_raw = f"temp_{split_name}_raw.txt"
    temp_dotted = f"temp_{split_name}_dotted.txt"

    if not os.path.exists(input_manifest):
        print(f"⏩ Skipping {split_name}: File not found.")
        continue

    print(f"\n--- Processing {split_name.upper()} Split ---")
    
    # Step 1: Extract and clean text
    print(f"📦 Loading {filename}...")
    with open(input_manifest, 'r', encoding='utf-8') as f:
        items = [json.loads(line) for line in f]

    with open(temp_raw, "w", encoding="utf-8") as f:
        for item in items:
            f.write(final_cleanup(item['text']) + "\n")

    # Step 2: Run Nakdimon Batch Diacritization
    print(f"🪄 Running Nakdimon on {len(items)} sentences...")
    try:
        subprocess.run([
            "python", "-m", "nakdimon", "predict", 
            temp_raw, temp_dotted
        ], check=True)
    except Exception as e:
        print(f"❌ Error in Nakdimon for {split_name}: {e}")
        continue

    # Step 3: Rebuild the manifest with fixed phonemes
    if os.path.exists(temp_dotted):
        with open(temp_dotted, "r", encoding="utf-8") as f:
            dotted_lines = f.read().splitlines()

        print(f"📝 Generating clean phonemes for {output_manifest}...")
        with open(output_manifest, 'w', encoding='utf-8') as f_out:
            for item, dotted in tqdm(zip(items, dotted_lines), total=len(items)):
                # Keep the original dotted text from Nakdimon in the text field
                item['text'] = dotted
                
                # Create a "friendly" version for the phonemizer
                text_for_tok = fix_for_phonemes(dotted)
                
                # Update phonemes based on the FIXED text
                try:
                    phonemes_result = text_tokenizer_he(text_for_tok)
                    raw_ph = " ".join(phonemes_result[0])
                    
                    # Final safety: remove any remaining (en) or (he) tags
                    clean_ph = re.sub(r'\(.*?\)', '', raw_ph)
                    item['phonemes'] = " ".join(clean_ph.split())
                except:
                    pass # Keep original if it fails
                
                f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
        
        # Cleanup temp files
        os.remove(temp_raw)
        os.remove(temp_dotted)
        print(f"✅ {split_name.upper()} is ready with clean phonemes!")

print("\n🏁 ALL DONE! Your entire dataset is now diacritized, normalized, and phonemized.")


--- Processing TRAIN Split ---
📦 Loading train_manifest_filtered.jsonl...
🪄 Running Nakdimon on 3632 sentences...


2026-03-08 00:48:00.457914: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-08 00:48:00.457972: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-08 00:48:00.458927: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-08 00:48:01.452080: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


2/2 [==============================] - 21s 5s/step
📝 Generating clean phonemes for ./voicecraft_data_nakdimon/manifest/train_manifest_nikud.jsonl...


100%|██████████| 3632/3632 [00:00<00:00, 3792.93it/s]


✅ TRAIN is ready with clean phonemes!

--- Processing VAL Split ---
📦 Loading val_manifest_filtered.jsonl...
🪄 Running Nakdimon on 294 sentences...


...

✅ VAL is ready with clean phonemes!

--- Processing TEST Split ---
📦 Loading test_manifest_filtered.jsonl...
🪄 Running Nakdimon on 84 sentences...



2026-03-08 00:48:46.146662: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-08 00:48:46.146726: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-08 00:48:46.147590: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-08 00:48:46.747193: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


1/1 [==============================] - 7s 7s/step
📝 Generating clean phonemes for ./voicecraft_data_nakdimon/manifest/test_manifest_nikud.jsonl...


100%|██████████| 84/84 [00:00<00:00, 3050.40it/s]

✅ TEST is ready with clean phonemes!

🏁 ALL DONE! Your entire dataset is now diacritized, normalized, and phonemized.


In [37]:
def convert_jsonl_to_voicecraft_txt(jsonl_input_path, txt_output_path, phonemes_base_dir):
    """
    Converts a JSONL manifest into the specific TXT format required by VoiceCraft
    and creates individual phoneme files for each sample.
    
    jsonl_input_path: Path to the source .jsonl file
    txt_output_path: Path where the final .txt manifest will be saved
    phonemes_base_dir: Directory where individual .txt phoneme files will be created
    """
    # Create phonemes directory if it doesn't exist
    os.makedirs(phonemes_base_dir, exist_ok=True)
    
    print(f"Processing: {jsonl_input_path} -> {txt_output_path}")
    
    samples_processed = 0
    with open(jsonl_input_path, 'r', encoding='utf-8') as f_in, \
         open(txt_output_path, 'w', encoding='utf-8') as f_out:
        
        for i, line in enumerate(f_in):
            data = json.loads(line)
            
            # Extract ID from audio_filepath (e.g., 'sample_0')
            item_id = os.path.basename(data['audio_filepath']).replace(".wav", "")
            duration = data.get('duration_frames', 0)
            phonemes = data.get('phonemes', "")
            
            # Create the individual phoneme file (e.g., ./phonemes/sample_0.txt)
            phn_file_path = os.path.join(phonemes_base_dir, f"{item_id}.txt")
            with open(phn_file_path, "w", encoding="utf-8") as f_phn:
                f_phn.write(phonemes)
            
            # Write to the manifest .txt (Format: Index <TAB> ID <TAB> Duration)
            f_out.write(f"{i}\t{item_id}\t{duration}\n")
            samples_processed += 1
            
    print(f"Successfully converted {samples_processed} samples.")

# --- Usage Example ---
# Define paths and call the function for train and validation
base_manifest_dir = "./voicecraft_data_nakdimon/manifest"
phn_dir = "./voicecraft_data_nakdimon/phonemes"

# For Train
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "train_manifest_nikud.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "train.txt"),
    phonemes_base_dir=phn_dir
)

# For Validation
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "val_manifest_nikud.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "validation.txt"),
    phonemes_base_dir=phn_dir
)

# For Test
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "test_manifest_nikud.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "test.txt"),
    phonemes_base_dir=phn_dir
)

Processing: ./voicecraft_data_nakdimon/manifest/train_manifest_nikud.jsonl -> ./voicecraft_data_nakdimon/manifest/train.txt
Successfully converted 3632 samples.
Processing: ./voicecraft_data_nakdimon/manifest/val_manifest_nikud.jsonl -> ./voicecraft_data_nakdimon/manifest/validation.txt
Successfully converted 294 samples.
Processing: ./voicecraft_data_nakdimon/manifest/test_manifest_nikud.jsonl -> ./voicecraft_data_nakdimon/manifest/test.txt
Successfully converted 84 samples.


In [ ]:
# Path to your final manifest
manifest_path = "./voicecraft_data_nakdimon/manifest/train_manifest_nikud.jsonl"

def check_random_samples(path, num_samples=5):
    if not os.path.exists(path):
        print(f"❌ File {path} not found!")
        return

    with open(path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    samples = random.sample(lines, min(num_samples, len(lines)))
    
    print(f"🔍 Checking {len(samples)} random samples from: {path}\n")
    print("-" * 50)
    
    for i, line in enumerate(samples, 1):
        item = json.loads(line)
        print(f"Sample #{i} (ID: {item.get('sample_id')})")
        print(f"📝 Text:     {item['text']}")
        print(f"🔊 Phonemes: {item['phonemes']}")
        
        # Check if there's any leftover English/hallucinations
        if "(" in item['phonemes'] or "h iː b ɹ uː" in item['phonemes']:
            print("⚠️ ALERT: English/Hallucination detected in phonemes!")
        else:
            print("✅ Phonemes look clean.")
        print("-" * 50)

# Run the check
check_random_samples(manifest_path)

🔍 Checking 5 random samples from: ./voicecraft_data_nakdimon/manifest/train_manifest_nikud.jsonl

--------------------------------------------------
Sample #1 (ID: 733)
📝 Text:     מוּקְדָּם יוֹתֵר הַשָּׁבוּעַ הִתְרַסְּקוּת מַסּוֹק מִשְׁטַרְתִּי הָרָגָה שְׁלוֹשָׁה אֲנָשִׁים וּפִצְעָה שְׁלוֹשָׁה נוֹסָפִים
🔊 Phonemes: m v k d a m _ j o t e ʁ _ a ʃ a v v a a _ i t ʁ a s k v t _ m a s o k _ m i ʃ t a ʁ t i j _ a ʁ a ɡ a ʔ _ ʃ l o ʃ a ʔ _ ʔ a n a ʃ i j m _ v f i t s o ʔ _ ʃ l o ʃ a ʔ _ n o s a f i j m
✅ Phonemes look clean.
--------------------------------------------------
Sample #2 (ID: 1278)
📝 Text:     הַסְבָּרָה הִיא שֶׁזּוֹ הַפַּעַם הַחֲמִישִׁית בַּהִיסְטוֹרְיָה שֶׁאֲנָשִׁים חָזוּ בְּמָה שֶׁהִתְגַּלָּה בְּסוֹפוֹ שֶׁל דָּבָר כְּחוֹמֶר מְאוּמַּת כִּימִית מֵהַמַּאְדִים שֶׁנּוֹפֵל לְכַדּוּר הָאָרֶץ
🔊 Phonemes: a s v a ʁ a ʔ _ i j ʔ _ ʃ e z o _ a f a a a m _ a χ a m i j ʃ i j t _ v a i j s t o ʁ j a ʔ _ ʃ e ʔ a n a ʃ i j m _ χ a z v _ v m a ʔ _ ʃ e i t ɡ a l a ʔ _ v s o f o _ ʃ e l _ d a v

In [30]:
# Quick test for the "vLo" issue
test_text = "וְלֹא לְכִיוון הָאָדָם"
# Ensure the tokenizer is in 'he' mode without English fallbacks
phonemes_test = text_tokenizer_he(test_text)
print(" ".join(phonemes_test[0]))

v a v _ ( en ) h iː b ɹ uː ʃ v ɑː ( he ) _ l a m e d _ ( en ) h iː b ɹ uː o ( he ) _ a l e f ʔ _ l χ i j w n _ a o d a m


In [31]:
# Test Lab: What does the tokenizer like?
test_cases = [
    "וְלֹא",          # 1. Standard (Vav + Shva + Lamed + Holam + Alef)
    "וְ לֹא",         # 2. Split with space
    "וְלֹא ",         # 3. With trailing space
    "וְלוֹ",          # 4. Vav-Lamed-Vav (Alternative spelling)
    "וְלֹּא",         # 5. With Dagesh on Lamed (Non-standard)
    "לֹא",           # 6. Just "Lo"
    "סֵפֶר",          # 7. Standard Sefer
    "סֵיפֶר"          # 8. Sefer with Yud
]

print("🔬 Running Tokenizer Test Lab...\n")
for i, text in enumerate(test_cases, 1):
    try:
        # Generate phonemes using your environment's tokenizer
        phonemes_result = text_tokenizer_he(text)
        res = " ".join(phonemes_result[0])
        print(f"Test {i}: [{text}] -> {res}")
    except Exception as e:
        print(f"Test {i}: [{text}] -> ❌ Error: {e}")

🔬 Running Tokenizer Test Lab...

Test 1: [וְלֹא] -> v a v _ ( en ) h iː b ɹ uː ʃ v ɑː ( he ) _ l a m e d _ ( en ) h iː b ɹ uː o ( he ) _ a l e f ʔ
Test 2: [וְ לֹא] -> v _ l a m e d _ ( en ) h iː b ɹ uː o ( he ) _ a l e f ʔ
Test 3: [וְלֹא ] -> v a v _ ( en ) h iː b ɹ uː ʃ v ɑː ( he ) _ l a m e d _ ( en ) h iː b ɹ uː o ( he ) _ a l e f ʔ
Test 4: [וְלוֹ] -> v l o
Test 5: [וְלֹּא] -> v a v _ ( en ) h iː b ɹ uː ʃ v ɑː ( he ) _ l a m e d _ ( en ) h iː b ɹ uː o _ h iː b ɹ uː d a ɡ ɛ ʃ ( he ) _ a l e f ʔ
Test 6: [לֹא] -> l a m e d _ ( en ) h iː b ɹ uː o ( he ) _ a l e f ʔ
Test 7: [סֵפֶר] -> s e f e ʁ
Test 8: [סֵיפֶר] -> s e j f e ʁ


In [32]:
# Laboratory 2: Isolating Vav and Alef
test_cases_2 = [
    "ו",      # Just Vav
    "וְ",     # Vav with Shva
    "א",      # Just Alef
    "אֳ",     # Alef with Hatef Qamats (common in 'Lo')
    "ולא",    # Plain text without nikud
    "לֹא",    # Lo with Holam Haser
    "לוֹא"    # Lo with Holam Male and Alef (rare spelling)
]

print("🔬 Running Isolation Test...\n")
for i, text in enumerate(test_cases_2, 1):
    try:
        phonemes_result = text_tokenizer_he(text)
        res = " ".join(phonemes_result[0])
        print(f"Test {i}: [{text}] -> {res}")
    except Exception as e:
        print(f"Test {i}: [{text}] -> ❌ Error: {e}")

🔬 Running Isolation Test...

Test 1: [ו] -> v a v
Test 2: [וְ] -> v
Test 3: [א] -> a l e f ʔ
Test 4: [אֳ] -> ʔ a
Test 5: [ולא] -> v l ʔ
Test 6: [לֹא] -> l a m e d _ ( en ) h iː b ɹ uː o ( he ) _ a l e f ʔ
Test 7: [לוֹא] -> l o ʔ


In [3]:
# --- Configuration ---
manifest_dir = "./voicecraft_data_merged/manifest"
splits = {
    "train": "train_manifest_filtered.jsonl",
    "val": "val_manifest_filtered.jsonl",
    "test": "test_manifest_filtered.jsonl"
}

# --- Helper Functions ---
def final_cleanup(text):
    """ Cleans text artifacts before diacritization """
    if not text: return ""
    return " ".join(text.replace('\\', '').replace('"', "'").split())

def fix_for_phonemes(text):
    """ 
    Normalizes Hebrew text specifically for the phonemizer to prevent 
    English descriptions (hallucinations) of letters and vowels.
    """
    if not text: return ""
    # 1. Global fix: Convert Holam Haser to Holam Male for all characters
    # Matches any char (except Vav) followed by Holam Haser and adds a Vav
    text = re.sub(r'([^ו])ֹ', r'\1וֹ', text)
    
    # 2. Specific fix for "Lo" (לֹא) which often fails
    text = text.replace("לֹא", "לוֹא")
    
    # 3. Clean up: Remove Dagesh and Shin/Sin dots that cause "spelling out" errors
    text = text.replace('ּ', '').replace('ׁ', '').replace('ׂ', '')
    
    return text

# Ensure the 'tests' directory exists (Nakdimon CLI requirement)
if not os.path.exists('tests'):
    os.makedirs('tests')

for split_name, filename in splits.items():
    input_manifest = os.path.join(manifest_dir, filename)
    output_manifest = os.path.join(manifest_dir, f"{split_name}_manifest_nikud.jsonl")
    
    temp_raw = f"temp_{split_name}_raw.txt"
    temp_dotted = f"temp_{split_name}_dotted.txt"

    if not os.path.exists(input_manifest):
        print(f"⏩ Skipping {split_name}: File not found.")
        continue

    print(f"\n--- Processing {split_name.upper()} Split ---")
    
    # Step 1: Extract and clean text
    print(f"📦 Loading {filename}...")
    with open(input_manifest, 'r', encoding='utf-8') as f:
        items = [json.loads(line) for line in f]

    with open(temp_raw, "w", encoding="utf-8") as f:
        for item in items:
            f.write(final_cleanup(item['text']) + "\n")

    # Step 2: Run Nakdimon Batch Diacritization
    print(f"🪄 Running Nakdimon on {len(items)} sentences...")
    try:
        subprocess.run([
            "python", "-m", "nakdimon", "predict", 
            temp_raw, temp_dotted
        ], check=True)
    except Exception as e:
        print(f"❌ Error in Nakdimon for {split_name}: {e}")
        continue

    # Step 3: Rebuild the manifest with fixed phonemes
    if os.path.exists(temp_dotted):
        with open(temp_dotted, "r", encoding="utf-8") as f:
            dotted_lines = f.read().splitlines()

        print(f"📝 Generating clean phonemes for {output_manifest}...")
        with open(output_manifest, 'w', encoding='utf-8') as f_out:
            for item, dotted in tqdm(zip(items, dotted_lines), total=len(items)):
                # Keep the original dotted text from Nakdimon in the text field
                item['text'] = dotted
                
                # Create a "friendly" version for the phonemizer
                text_for_tok = fix_for_phonemes(dotted)
                
                # Update phonemes based on the FIXED text
                try:
                    phonemes_result = text_tokenizer_he(text_for_tok)
                    raw_ph = " ".join(phonemes_result[0])
                    
                    # Final safety: remove any remaining (en) or (he) tags
                    clean_ph = re.sub(r'\(.*?\)', '', raw_ph)
                    item['phonemes'] = " ".join(clean_ph.split())
                except:
                    pass # Keep original if it fails
                
                f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
        
        # Cleanup temp files
        os.remove(temp_raw)
        os.remove(temp_dotted)
        print(f"✅ {split_name.upper()} is ready with clean phonemes!")

print("\n🏁 ALL DONE! Your entire dataset is now diacritized, normalized, and phonemized.")


--- Processing TRAIN Split ---
📦 Loading train_manifest_filtered.jsonl...
🪄 Running Nakdimon on 11428 sentences...


2026-03-14 14:02:22.405038: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-14 14:02:22.822016: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-14 14:02:22.822150: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-14 14:02:22.886862: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-14 14:02:23.018062: I tensorflow/core/platform/cpu_feature_guar

3/3 [==============================] - 36s 9s/step
📝 Generating clean phonemes for ./voicecraft_data_merged/manifest/train_manifest_nikud.jsonl...


100%|██████████| 11428/11428 [00:01<00:00, 6365.88it/s]


✅ TRAIN is ready with clean phonemes!

--- Processing VAL Split ---
📦 Loading val_manifest_filtered.jsonl...
🪄 Running Nakdimon on 2240 sentences...


...

✅ VAL is ready with clean phonemes!

--- Processing TEST Split ---
📦 Loading test_manifest_filtered.jsonl...
🪄 Running Nakdimon on 84 sentences...


2026-03-14 14:03:32.710904: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-14 14:03:32.748520: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-14 14:03:32.748592: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-14 14:03:32.749645: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-14 14:03:32.755131: I tensorflow/core/platform/cpu_feature_guar

1/1 [==============================] - 10s 10s/step
📝 Generating clean phonemes for ./voicecraft_data_merged/manifest/test_manifest_nikud.jsonl...


100%|██████████| 84/84 [00:00<00:00, 4436.74it/s]

✅ TEST is ready with clean phonemes!

🏁 ALL DONE! Your entire dataset is now diacritized, normalized, and phonemized.


In [4]:
def convert_jsonl_to_voicecraft_txt(jsonl_input_path, txt_output_path, phonemes_base_dir):
    """
    Converts a JSONL manifest into the specific TXT format required by VoiceCraft
    and creates individual phoneme files for each sample.
    
    jsonl_input_path: Path to the source .jsonl file
    txt_output_path: Path where the final .txt manifest will be saved
    phonemes_base_dir: Directory where individual .txt phoneme files will be created
    """
    # Create phonemes directory if it doesn't exist
    os.makedirs(phonemes_base_dir, exist_ok=True)
    
    print(f"Processing: {jsonl_input_path} -> {txt_output_path}")
    
    samples_processed = 0
    with open(jsonl_input_path, 'r', encoding='utf-8') as f_in, \
         open(txt_output_path, 'w', encoding='utf-8') as f_out:
        
        for i, line in enumerate(f_in):
            data = json.loads(line)
            
            # Extract ID from audio_filepath (e.g., 'sample_0')
            item_id = os.path.basename(data['audio_filepath']).replace(".wav", "")
            duration = data.get('duration_frames', 0)
            phonemes = data.get('phonemes', "")
            
            # Create the individual phoneme file (e.g., ./phonemes/sample_0.txt)
            phn_file_path = os.path.join(phonemes_base_dir, f"{item_id}.txt")
            with open(phn_file_path, "w", encoding="utf-8") as f_phn:
                f_phn.write(phonemes)
            
            # Write to the manifest .txt (Format: Index <TAB> ID <TAB> Duration)
            f_out.write(f"{i}\t{item_id}\t{duration}\n")
            samples_processed += 1
            
    print(f"Successfully converted {samples_processed} samples.")

# --- Usage Example ---
# Define paths and call the function for train and validation
base_manifest_dir = "./voicecraft_data_merged/manifest"
phn_dir = "./voicecraft_data_merged/phonemes"

# For Train
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "train_manifest_nikud.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "train.txt"),
    phonemes_base_dir=phn_dir
)

# For Validation
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "val_manifest_nikud.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "validation.txt"),
    phonemes_base_dir=phn_dir
)

# For Test
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "test_manifest_nikud.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "test.txt"),
    phonemes_base_dir=phn_dir
)

Processing: ./voicecraft_data_merged/manifest/train_manifest_nikud.jsonl -> ./voicecraft_data_merged/manifest/train.txt
Successfully converted 11428 samples.
Processing: ./voicecraft_data_merged/manifest/val_manifest_nikud.jsonl -> ./voicecraft_data_merged/manifest/validation.txt
Successfully converted 2240 samples.
Processing: ./voicecraft_data_merged/manifest/test_manifest_nikud.jsonl -> ./voicecraft_data_merged/manifest/test.txt
Successfully converted 84 samples.


In [5]:
# Path to your final manifest
manifest_path = "./voicecraft_data_merged/manifest/train_manifest_nikud.jsonl"

def check_random_samples(path, num_samples=5):
    if not os.path.exists(path):
        print(f"❌ File {path} not found!")
        return

    with open(path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    samples = random.sample(lines, min(num_samples, len(lines)))
    
    print(f"🔍 Checking {len(samples)} random samples from: {path}\n")
    print("-" * 50)
    
    for i, line in enumerate(samples, 1):
        item = json.loads(line)
        print(f"Sample #{i} (ID: {item.get('sample_id')})")
        print(f"📝 Text:     {item['text']}")
        print(f"🔊 Phonemes: {item['phonemes']}")
        
        # Check if there's any leftover English/hallucinations
        if "(" in item['phonemes'] or "h iː b ɹ uː" in item['phonemes']:
            print("⚠️ ALERT: English/Hallucination detected in phonemes!")
        else:
            print("✅ Phonemes look clean.")
        print("-" * 50)

# Run the check
check_random_samples(manifest_path)

🔍 Checking 5 random samples from: ./voicecraft_data_merged/manifest/train_manifest_nikud.jsonl

--------------------------------------------------
Sample #1 (ID: 3730)
📝 Text:     כְּנֵסִיּוֹת מָסוֹרְתִּיּוֹת יוֹתֵר בְּמִקְרִים רַבִּים מְקַיְּימוֹת לֵיל שִׁימּוּרִים שֶׁל חַג הַפַּסְחָא בְּמוֹצָאֵי שַׁבָּת בְּסוֹף הַשָּׁבוּעַ שֶׁל הַפַּסְחָא וְהַקְּהִילּוֹת נוֹהֲגוֹת לִפְצוֹחַ בַּחֲגִיגוֹת בְּדִיּוּק בַּחֲצוֹת לְרֶגֶל תְּחִיַּית יֵשׁוּ
🔊 Phonemes: χ n e s i j o t _ m a s o ʁ t i j o t _ j o t e ʁ _ v m i k ʁ i j m _ ʁ a v i j m _ m k a j j m o t _ l e j l _ ʃ i j m v ʁ i j m _ ʃ e l _ a χ ɡ _ a f a s χ a ʔ _ v m o t s a e j _ ʃ a v a t _ v s o f _ a ʃ a v v a a _ ʃ e l _ a f a s χ a ʔ _ v a k i j l o t _ n o ʔ a ɡ o t _ l i f t s o a χ _ v a χ a ɡ i j ɡ o t _ v d i j v k _ v a χ a t s o t _ l ʁ e ɡ e l _ t χ i j a j t _ j e ʃ v
✅ Phonemes look clean.
--------------------------------------------------
Sample #2 (ID: 1670)
📝 Text:     לְמַעֲשֶׂה לַקּוֹדִים הָאֲזוֹרִיִּים אֵין כָּל הַשְׁפּ

In [7]:
def filter_and_clean_manifest(input_path, output_path):
    """
    Cleans text and filters out samples that are likely noisy or mismatched.
    Keeps the original sample_id for the remaining samples.
    """
    def is_bad_sample(text):
        # 1. Filter out lines that still have Latin characters (English) 
        # after basic cleaning, as they often don't match the Hebrew audio.
        if re.search(r'[a-zA-Z]', text):
            return True
        # # 2. Filter out very short sentences (less than 3 words) which are often noise
        # if len(text.split()) < 3:
        #     return True
        return False

    def clean_line(text):
        # Remove backslashes and replace double quotes with single ones
        text = text.replace('\\', '').replace('"', "'")
        # Remove technical time-zone noise (UTC/GMT)
        text = re.sub(r'(?i)utc[+-]?\d*', '', text)
        text = re.sub(r'(?i)gmt[+-]?\d*', '', text)
        # Remove isolated dagesh or other non-attached Hebrew diacritics
        text = text.replace('ּ', '') 
        return " ".join(text.split())

    count_before = 0
    count_after = 0

    if not os.path.exists(input_path):
        print(f"❌ Error: Input file {input_path} not found!")
        return

    with open(input_path, 'r', encoding='utf-8') as f_in, \
         open(output_path, 'w', encoding='utf-8') as f_out:
        
        for line in tqdm(f_in, desc=f"Processing {os.path.basename(input_path)}"):
            count_before += 1
            try:
                item = json.loads(line)
                
                # 1. Initial cleaning
                cleaned_text = clean_line(item['text'])
                
                # 2. Check if we should keep this sample
                if is_bad_sample(cleaned_text):
                    continue # Skip this sample entirely
                
                # 3. Update text and phonemes
                item['text'] = cleaned_text
                # Regenerate phonemes (must be sync'd with cleaned text)
                phonemes_result = text_tokenizer_he(cleaned_text)
                final_phonemes = " ".join(phonemes_result[0])
                
                # NEW: Double check the phonemes for hallucinations
                if re.search(r'[a-zA-Z]', final_phonemes):
                      print(f"⚠️ Hallucination detected in phonemes for ID {item.get('sample_id')}, skipping.")
                      continue # Skip this sample entirely
                
                item['phonemes'] = final_phonemes
                
                # 4. Save entry
                f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
                count_after += 1
            except Exception as e:
                print(f"Error on line {count_before}: {e}")

    print(f"\n📊 Process finished for {os.path.basename(input_path)}!")
    print(f"Original samples: {count_before}")
    print(f"Cleaned samples kept: {count_after}")
    print(f"Dropped: {count_before - count_after} samples")

# --- Execution ---
base_dir = "./voicecraft_data_merged"
manifest_dir = os.path.join(base_dir, "manifest")

# Fixing the path string formatting
train_input = os.path.join(manifest_dir, "train_manifest_nikud.jsonl")
train_output = os.path.join(manifest_dir, "train_manifest_nikud_filtered.jsonl")

val_input = os.path.join(manifest_dir, "val_manifest_nikud.jsonl")
val_output = os.path.join(manifest_dir, "val_manifest_nikud_filtered.jsonl")

test_input = os.path.join(manifest_dir, "test_manifest_nikud.jsonl")
test_output = os.path.join(manifest_dir, "test_manifest_nikud_filtered.jsonl")

# Run for train and val
filter_and_clean_manifest(train_input, train_output)
filter_and_clean_manifest(val_input, val_output)
filter_and_clean_manifest(test_input, test_output)

Processing train_manifest_nikud.jsonl: 738it [00:00, 7363.56it/s]

⚠️ Hallucination detected in phonemes for ID 0, skipping.
⚠️ Hallucination detected in phonemes for ID 1, skipping.
⚠️ Hallucination detected in phonemes for ID 2, skipping.
⚠️ Hallucination detected in phonemes for ID 3, skipping.
⚠️ Hallucination detected in phonemes for ID 4, skipping.
⚠️ Hallucination detected in phonemes for ID 5, skipping.
⚠️ Hallucination detected in phonemes for ID 6, skipping.
⚠️ Hallucination detected in phonemes for ID 7, skipping.
⚠️ Hallucination detected in phonemes for ID 8, skipping.
⚠️ Hallucination detected in phonemes for ID 9, skipping.
⚠️ Hallucination detected in phonemes for ID 10, skipping.
⚠️ Hallucination detected in phonemes for ID 11, skipping.
⚠️ Hallucination detected in phonemes for ID 12, skipping.
⚠️ Hallucination detected in phonemes for ID 13, skipping.
⚠️ Hallucination detected in phonemes for ID 14, skipping.
⚠️ Hallucination detected in phonemes for ID 15, skipping.
⚠️ Hallucination detected in phonemes for ID 16, skipping.
⚠️ Hall

Processing train_manifest_nikud.jsonl: 2244it [00:00, 7177.69it/s]

⚠️ Hallucination detected in phonemes for ID 1464, skipping.
⚠️ Hallucination detected in phonemes for ID 1465, skipping.
⚠️ Hallucination detected in phonemes for ID 1466, skipping.
⚠️ Hallucination detected in phonemes for ID 1467, skipping.
⚠️ Hallucination detected in phonemes for ID 1468, skipping.
⚠️ Hallucination detected in phonemes for ID 1469, skipping.
⚠️ Hallucination detected in phonemes for ID 1470, skipping.
⚠️ Hallucination detected in phonemes for ID 1471, skipping.
⚠️ Hallucination detected in phonemes for ID 1472, skipping.
⚠️ Hallucination detected in phonemes for ID 1473, skipping.
⚠️ Hallucination detected in phonemes for ID 1474, skipping.
⚠️ Hallucination detected in phonemes for ID 1475, skipping.
⚠️ Hallucination detected in phonemes for ID 1476, skipping.
⚠️ Hallucination detected in phonemes for ID 1477, skipping.
⚠️ Hallucination detected in phonemes for ID 1478, skipping.
⚠️ Hallucination detected in phonemes for ID 1479, skipping.
⚠️ Hallucination detecte

Processing train_manifest_nikud.jsonl: 3766it [00:00, 7444.68it/s]

...

⚠️ Hallucination detected in phonemes for ID 9370, skipping.
⚠️ Hallucination detected in phonemes for ID 9371, skipping.
⚠️ Hallucination detected in phonemes for ID 9372, skipping.
⚠️ Hallucination detected in phonemes for ID 9373, skipping.
⚠️ Hallucination detected in phonemes for ID 9374, skipping.
⚠️ Hallucination detected in phonemes for ID 9375, skipping.
⚠️ Hallucination detected in phonemes for ID 9376, skipping.
⚠️ Hallucination detected in phonemes for ID 9377, skipping.
⚠️ Hallucination detected in phonemes for ID 9378, skipping.
⚠️ Hallucination detected in phonemes for ID 9379, skipping.
⚠️ Hallucination detected in phonemes for ID 9380, skipping.
⚠️ Hallucination detected in phonemes for ID 9381, skipping.
⚠️ Hallucination detected in phonemes for ID 9383, skipping.
⚠️ Hallucination detected in phonemes for ID 9384, skipping.
⚠️ Hallucination detected in phonemes for ID 9385, skipping.
⚠️ Hallucination detected in phonemes for ID 9386, skipping.
⚠️ Hallucination detecte

Processing val_manifest_nikud.jsonl: 2240it [00:00, 6329.06it/s]


⚠️ Hallucination detected in phonemes for ID 4257, skipping.
⚠️ Hallucination detected in phonemes for ID 4258, skipping.
⚠️ Hallucination detected in phonemes for ID 4259, skipping.
⚠️ Hallucination detected in phonemes for ID 4260, skipping.
⚠️ Hallucination detected in phonemes for ID 4261, skipping.
⚠️ Hallucination detected in phonemes for ID 4262, skipping.
⚠️ Hallucination detected in phonemes for ID 4263, skipping.
⚠️ Hallucination detected in phonemes for ID 4264, skipping.
⚠️ Hallucination detected in phonemes for ID 4265, skipping.
⚠️ Hallucination detected in phonemes for ID 4266, skipping.
⚠️ Hallucination detected in phonemes for ID 4267, skipping.
⚠️ Hallucination detected in phonemes for ID 4268, skipping.
⚠️ Hallucination detected in phonemes for ID 4269, skipping.

📊 Process finished for val_manifest_nikud.jsonl!
Original samples: 2240
Cleaned samples kept: 0
Dropped: 2240 samples


Processing test_manifest_nikud.jsonl: 84it [00:00, 2847.46it/s]

⚠️ Hallucination detected in phonemes for ID 4270, skipping.
⚠️ Hallucination detected in phonemes for ID 4271, skipping.
⚠️ Hallucination detected in phonemes for ID 4272, skipping.
⚠️ Hallucination detected in phonemes for ID 4273, skipping.
⚠️ Hallucination detected in phonemes for ID 4274, skipping.
⚠️ Hallucination detected in phonemes for ID 4275, skipping.
⚠️ Hallucination detected in phonemes for ID 4276, skipping.
⚠️ Hallucination detected in phonemes for ID 4277, skipping.
⚠️ Hallucination detected in phonemes for ID 4278, skipping.
⚠️ Hallucination detected in phonemes for ID 4279, skipping.
⚠️ Hallucination detected in phonemes for ID 4280, skipping.
⚠️ Hallucination detected in phonemes for ID 4282, skipping.
⚠️ Hallucination detected in phonemes for ID 4283, skipping.
⚠️ Hallucination detected in phonemes for ID 4284, skipping.
⚠️ Hallucination detected in phonemes for ID 4285, skipping.
⚠️ Hallucination detected in phonemes for ID 4287, skipping.
⚠️ Hallucination detecte

In [8]:
import json
import re

# Load just one line from your NIKUD file
test_file = "./voicecraft_data_merged/manifest/train_manifest_nikud.jsonl"

with open(test_file, 'r', encoding='utf-8') as f:
    line = f.readline()
    item = json.loads(line)
    
    text = item['text']
    # We'll simulate the phonemization result here since we want to see what's in the file
    phonemes = item.get('phonemes', '')

    print(f"--- Debugging ID {item.get('sample_id')} ---")
    print(f"Original Text: {text}")
    print(f"Phonemes in file: {phonemes}")
    
    # Check Step 1: Latin in Text
    if re.search(r'[a-zA-Z]', text):
        print("❌ Rejected because of English in TEXT")
    
    # Check Step 2: Length
    if len(text.split()) < 3:
        print("❌ Rejected because it's too SHORT")
        
    # Check Step 3: Latin in Phonemes
    if re.search(r'[a-zA-Z]', str(phonemes)):
        print(f"❌ Rejected because of English in PHONEMES")
        # Let's see EXACTLY what Latin characters are there:
        found = re.findall(r'[a-zA-Z]', str(phonemes))
        print(f"Found these Latin chars: {set(found)}")

--- Debugging ID 0 ---
Original Text: הִיא מְבִינָה אוֹתִי יוֹתֵר מִכָּל אֶחָד אַחֵר
Phonemes in file: i j ʔ _ m v i j n a ʔ _ o t i j _ j o t e ʁ _ m i χ a l _ e χ a d _ a χ e ʁ
❌ Rejected because of English in PHONEMES
Found these Latin chars: {'o', 'd', 'a', 't', 'e', 'i', 'n', 'm', 'v', 'l', 'j'}


In [9]:
def filter_and_clean_manifest(input_path, output_path):
    # רשימת ה"הזיות" המוכרות - אם אחת מאלה מופיעה בפונמות, הדגימה נזרקת
    bad_phoneme_markers = [
        "(", ")",           # סוגריים שהפונמייזר מוסיף להערות
        "h iː b ɹ uː",      # המילה Hebrew
        "a l e f",          # שם האות א' באנגלית
        "m e m",            # שם האות מ' באנגלית
        "v a v",            # שם האות ו' באנגלית
        "b a c k t i c k",  # תיאור תווים טכניים
        "l e t t e r"       # המילה Letter
    ]

    count_before = 0
    count_after = 0

    with open(input_path, 'r', encoding='utf-8') as f_in, \
         open(output_path, 'w', encoding='utf-8') as f_out:
        
        for line in tqdm(f_in, desc=f"Filtering {os.path.basename(input_path)}"):
            count_before += 1
            item = json.loads(line)
            
            # 1. ניקוי טקסט בסיסי (כמו שעשינו קודם)
            text = item['text'].replace('\\', '').replace('"', "'").strip()
            
            # 2. סינון טקסט מקורי (אם יש אנגלית בטקסט עצמו או משפט קצר מדי)
            if re.search(r'[a-zA-Z]', text) or len(text.split()) < 3:
                continue

            # 3. הבדיקה המנצחת שלך על הפונמות
            phonemes = item.get('phonemes', '')
            is_hallucination = any(marker in phonemes for marker in bad_phoneme_markers)
            
            if is_hallucination:
                # מדפיס רק כדי שתראי מה נזרק (אפשר לבטל אם זה יותר מדי)
                # print(f"⚠️ Hallucination ID {item.get('sample_id')} removed.")
                continue

            # 4. אם הכל תקין - שומרים לקובץ החדש
            item['text'] = text
            f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
            count_after += 1

    print(f"\n📊 Cleanup Finished!")
    print(f"Kept: {count_after} | Dropped: {count_before - count_after}")

# --- Execution ---
base_dir = "./voicecraft_data_merged"
manifest_dir = os.path.join(base_dir, "manifest")

# Fixing the path string formatting
train_input = os.path.join(manifest_dir, "train_manifest_nikud.jsonl")
train_output = os.path.join(manifest_dir, "train_manifest_nikud_filtered.jsonl")

val_input = os.path.join(manifest_dir, "val_manifest_nikud.jsonl")
val_output = os.path.join(manifest_dir, "val_manifest_nikud_filtered.jsonl")

test_input = os.path.join(manifest_dir, "test_manifest_nikud.jsonl")
test_output = os.path.join(manifest_dir, "test_manifest_nikud_filtered.jsonl")

# Run for train and val
filter_and_clean_manifest(train_input, train_output)
filter_and_clean_manifest(val_input, val_output)
filter_and_clean_manifest(test_input, test_output)   

Filtering train_manifest_nikud.jsonl: 11428it [00:00, 89073.02it/s]



📊 Cleanup Finished!
Kept: 10071 | Dropped: 1357


Filtering val_manifest_nikud.jsonl: 2240it [00:00, 85293.42it/s]



📊 Cleanup Finished!
Kept: 2071 | Dropped: 169


Filtering test_manifest_nikud.jsonl: 84it [00:00, 23131.87it/s]


📊 Cleanup Finished!
Kept: 71 | Dropped: 13


In [10]:
# For Train
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "train_manifest_nikud_filtered.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "train.txt"),
    phonemes_base_dir=phn_dir
)

# For Validation
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "val_manifest_nikud_filtered.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "validation.txt"),
    phonemes_base_dir=phn_dir
)

# For Test
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "test_manifest_nikud_filtered.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "test.txt"),
    phonemes_base_dir=phn_dir
)

Processing: ./voicecraft_data_merged/manifest/train_manifest_nikud_filtered.jsonl -> ./voicecraft_data_merged/manifest/train.txt
Successfully converted 10071 samples.
Processing: ./voicecraft_data_merged/manifest/val_manifest_nikud_filtered.jsonl -> ./voicecraft_data_merged/manifest/validation.txt
Successfully converted 2071 samples.
Processing: ./voicecraft_data_merged/manifest/test_manifest_nikud_filtered.jsonl -> ./voicecraft_data_merged/manifest/test.txt
Successfully converted 71 samples.


In [11]:
# Define your dataset details
user_slug = "daniellasolo"
dataset_slug = "voicecraft-hebrew-merged-nakdimon"

metadata = {
  "title": "VoiceCraft Hebrew Processed Data",
  "id": f"{user_slug}/{dataset_slug}",
  "licenses": [{"name": "CC0-1.0"}]
}

# Save metadata file
with open('./voicecraft_data_merged/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

print("✅ Metadata file created!")

✅ Metadata file created!


In [4]:
!kaggle datasets create -p ./voicecraft_data_merged --dir-mode zip

Starting upload for file phonemes.zip
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/blobs.BlobApiService/StartBlobUpload


In [3]:
print(f"User: {os.environ.get('KAGGLE_USERNAME')}")

User: None


In [4]:
# --- Configuration ---
manifest_dir = "./voicecraft_data_merged/manifest"
splits = {
    "train": "train_manifest_filtered.jsonl",
    "val": "val_manifest_filtered.jsonl",
    "test": "test_manifest_filtered.jsonl"
}

# Ensure the 'tests' directory exists (Nakdimon CLI requirement)
if not os.path.exists('tests'):
    os.makedirs('tests')

for split_name, filename in splits.items():
    input_manifest = os.path.join(manifest_dir, filename)
    output_manifest = os.path.join(manifest_dir, f"{split_name}_manifest_nikud_2.jsonl")
    
    temp_raw = f"temp_{split_name}_raw.txt"
    temp_dotted = f"temp_{split_name}_dotted.txt"

    if not os.path.exists(input_manifest):
        print(f"⏩ Skipping {split_name}: File not found.")
        continue

    print(f"\n--- Processing {split_name.upper()} Split ---")
    
    # Step 1: Extract text AS IS
    print(f"📦 Loading {filename}...")
    with open(input_manifest, 'r', encoding='utf-8') as f:
        items = [json.loads(line) for line in f]

    with open(temp_raw, "w", encoding="utf-8") as f:
        for item in items:
            # כתיבת הטקסט המקורי ללא ניקוי
            f.write(item['text'] + "\n")

    # Step 2: Run Nakdimon Batch Diacritization
    print(f"🪄 Running Nakdimon on {len(items)} sentences...")
    try:
        subprocess.run([
            "python", "-m", "nakdimon", "predict", 
            temp_raw, temp_dotted
        ], check=True)
    except Exception as e:
        print(f"❌ Error in Nakdimon for {split_name}: {e}")
        continue

    # Step 3: Rebuild the manifest with dotted text
    if os.path.exists(temp_dotted):
        with open(temp_dotted, "r", encoding="utf-8") as f:
            dotted_lines = f.read().splitlines()

        print(f"📝 Saving diacritized text to {output_manifest}...")
        with open(output_manifest, 'w', encoding='utf-8') as f_out:
            for item, dotted in zip(items, dotted_lines):
                # עדכון שדה הטקסט לטקסט המנוקד בלבד
                item['text'] = dotted
                f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
        
        # Cleanup temp files
        os.remove(temp_raw)
        os.remove(temp_dotted)
        print(f"✅ {split_name.upper()} is ready!")

print("\n🏁 ALL DONE! Your dataset is now diacritized.")


--- Processing TRAIN Split ---
📦 Loading train_manifest_filtered.jsonl...
🪄 Running Nakdimon on 11428 sentences...


2026-03-21 00:03:19.428114: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 00:03:19.959117: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-21 00:03:19.959370: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 00:03:20.040343: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-21 00:03:20.218935: I tensorflow/core/platform/cpu_feature_guar

3/3 [==============================] - 34s 9s/step
📝 Saving diacritized text to ./voicecraft_data_merged/manifest/train_manifest_nikud_2.jsonl...
✅ TRAIN is ready!

--- Processing VAL Split ---
📦 Loading val_manifest_filtered.jsonl...
🪄 Running Nakdimon on 2240 sentences...


2026-03-21 00:04:10.186408: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 00:04:10.228445: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-21 00:04:10.228518: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 00:04:10.230093: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-21 00:04:10.237173: I tensorflow/core/platform/cpu_feature_guar

1/1 [==============================] - 11s 11s/step
📝 Saving diacritized text to ./voicecraft_data_merged/manifest/val_manifest_nikud_2.jsonl...
✅ VAL is ready!

--- Processing TEST Split ---
📦 Loading test_manifest_filtered.jsonl...
🪄 Running Nakdimon on 84 sentences...


2026-03-21 00:04:27.769521: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 00:04:27.807831: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-21 00:04:27.807901: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 00:04:27.809244: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-21 00:04:27.814937: I tensorflow/core/platform/cpu_feature_guar

1/1 [==============================] - 8s 8s/step
📝 Saving diacritized text to ./voicecraft_data_merged/manifest/test_manifest_nikud_2.jsonl...
✅ TEST is ready!

🏁 ALL DONE! Your dataset is now diacritized.


In [5]:
def create_vocalized_vocab(old_vocab_path, new_vocab_path):
    # 1. קריאת ה-Vocab הקיים (זה עם האנגלית והפונמות IPA)
    with open(old_vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f.readlines() if line.strip()]
    
    # 2. הגדרת התווים החדשים שאנחנו מוסיפים
    hebrew_letters = list("אבגדהוזחטיכלמנסעפצקרשתךםןףץ")
    
    # סימני ניקוד ב-Unicode (הדרך הכי בטוחה לייצג אותם)
    hebrew_nikud = [
        '\u05b0', # שווא
        '\u05b1', # חטף סגול
        '\u05b2', # חטף פתח
        '\u05b3', # חטף קמץ
        '\u05b4', # חיריק
        '\u05b5', # צירי
        '\u05b6', # סגול
        '\u05b7', # פתח
        '\u05b8', # קמץ
        '\u05b9', # חולם
        '\u05bb', # קובוץ
        '\u05bc', # דגש/שורוק
        '\u05c1', # שין ימנית
        '\u05c2', # שין שמאלית
    ]
    
    punctuation = list(".,?!:;-\" ")

    # 3. הוספה לסוף הרשימה רק אם התו לא קיים כבר
    added_count = 0
    for char in (hebrew_letters + hebrew_nikud + punctuation):
        if char not in vocab:
            vocab.append(char)
            added_count += 1
            
    # 4. שמירה לקובץ החדש
    with open(new_vocab_path, 'w', encoding='utf-8') as f:
        for token in vocab:
            f.write(f"{token}\n")
            
    print(f"הסתיים! נוספו {added_count} תווים חדשים.")
    print(f"גודל ה-Vocab הסופי (זה המספר שצריך לעדכן ב-Config): {len(vocab)}")

# הרצה (תחליפי לשמות הקבצים שלך)
create_vocalized_vocab("voicecraft_data_merged/vocab.txt", "voicecraft_data_merged/vocab_new.txt")

הסתיים! נוספו 50 תווים חדשים.
גודל ה-Vocab הסופי (זה המספר שצריך לעדכן ב-Config): 154


In [7]:
import os

def create_hebrew_vocalized_vocab(old_vocab_path, new_vocab_path):
    # 1. טעינת ה-Vocab הקיים (ה-IPA והאנגלית)
# 1. קריאת ה-Vocab הקיים
    with open(old_vocab_path, 'r', encoding='utf-8') as f:
        # אנחנו לוקחים רק את התו/פונמה מכל שורה (מתעלמים מהמספר הקיים אם יש)
        lines = [line.strip() for line in f.readlines() if line.strip()]
    
    existing_tokens = []
    for line in lines:
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            existing_tokens.append(parts[1])
        else:
            existing_tokens.append(parts[0])

    # 2. הגדרת התוספות העבריות
    hebrew_alphabet = list("אבגדהוזחטיכלמנסעפצקרשתךםןףץ")
    
    # ניקוד (Unicode)
    hebrew_nikud = [
        '\u05b0', '\u05b1', '\u05b2', '\u05b3', '\u05b4', '\u05b5', 
        '\u05b6', '\u05b7', '\u05b8', '\u05b9', '\u05bb', '\u05bc', 
        '\u05c1', '\u05c2'
    ]
    
    punctuation = [".", ",", "?", "!", ":", ";", "-", '"']

    # 3. איחוד הכל (שמירה על סדר: אנגלית/IPA קודם, אז עברית)
    all_tokens = existing_tokens.copy()
    for char in (hebrew_alphabet + hebrew_nikud + punctuation):
        if char not in all_tokens:
            all_tokens.append(char)

    # 4. כתיבה לקובץ בפורמט ממוספר: "index token"
    with open(new_vocab_path, 'w', encoding='utf-8') as f:
        for i, token in enumerate(all_tokens):
            f.write(f"{i} {token}\n")
    
    print(f"בוצע! הקובץ {new_vocab_path} נוצר עם {len(all_tokens)} שורות ממוספרות.")
    print(f"האות א' קיבלה את אינדקס: {all_tokens.index('א')}")
    return all_tokens

# --- פונקציית בדיקה לראות שהכל עובד ---
def test_tokenizer(vocab_list, text):
    token_to_id = {token: i for i, token in enumerate(vocab_list)}
    ids = [token_to_id.get(char, "UNK") for char in text]
    print(f"--- בדיקת טוקנייזר ---")
    print(f"טקסט לבדיקה: {text}")
    print(f"תווים: {list(text)}")
    print(f"אינדקסים: {ids}")

# הרצה:
# 1. ודאי שיש לך קובץ בשם vocab_old.txt עם ה-IPA ששלחת לי
my_vocab = create_vocalized_vocab("voicecraft_data_merged/vocab.txt", "voicecraft_data_merged/vocab_new.txt")

# 2. בדיקה על המילה "בָּ" (בית עם דגש וקמץ)
if my_vocab:
    test_tokenizer(my_vocab, "בָּ")

הסתיים! נוספו 50 תווים חדשים.
גודל ה-Vocab הסופי (זה המספר שצריך לעדכן ב-Config): 154


In [9]:
def append_hebrew_to_existing_vocab(input_file, output_file):
# 1. קריאת הקובץ הקיים (אנגלית + IPA)
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]
    
    # מציאת האינדקס האחרון
    last_index = -1
    if lines:
        try:
            last_index = int(lines[-1].split()[0])
        except (ValueError, IndexError):
            last_index = len(lines) - 1
    
    next_idx = last_index + 1
    
    # 2. הגדרת התווים להוספה
    hebrew_alphabet = list("אבגדהוזחטיכלמנסעפצקרשתךםןףץ")
    # ניקוד - שימוש בקידוד Unicode ישיר כדי למנוע "קפיצות" ויזואליות בסקריפט
    hebrew_nikud = [
        '\u05b0', '\u05b1', '\u05b2', '\u05b3', '\u05b4', '\u05b5', 
        '\u05b6', '\u05b7', '\u05b8', '\u05b9', '\u05bb', '\u05bc', 
        '\u05c1', '\u05c2'
    ]
    punctuation = [".", ",", "?", "!", ":", ";", "-", '"', " "]

    new_tokens = hebrew_alphabet + hebrew_nikud + punctuation

    # 3. כתיבה לקובץ החדש
    with open(output_file, 'w', encoding='utf-8') as f:
        # קודם כל מעתיקים את המקור
        for line in lines:
            f.write(line + '\n')
        
        # מוסיפים את החדשים עם רווח מובטח
        for token in new_tokens:
            # בדיקה אם התו כבר קיים (כדי לא לייצר כפילויות)
            existing_chars = [l.split(maxsplit=1)[1] for l in lines if len(l.split()) > 1]
            if token in existing_chars:
                continue
                
            # כתיבה פורמלית: {מספר} {רווח} {תו}
            f.write(f"{next_idx} {token}\n")
            next_idx += 1

    print(f"הקובץ סודר! סה\"כ טוקנים: {next_idx}")
    print(f"אינדקס אחרון: {next_idx - 1}")

# שימוש:
append_hebrew_to_existing_vocab("voicecraft_data_merged/vocab.txt", "voicecraft_data_merged/vocab_new.txt")


הקובץ סודר! סה"כ טוקנים: 147
אינדקס אחרון: 146


In [10]:
def fix_and_append_vocalized_vocab(input_file, output_file):
    # 1. קריאת הקובץ הקיים (אנגלית + IPA)
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]
    
    # מציאת האינדקס האחרון
    last_index = -1
    if lines:
        try:
            last_index = int(lines[-1].split()[0])
        except (ValueError, IndexError):
            last_index = len(lines) - 1
    
    next_idx = last_index + 1
    
    # 2. הגדרת התווים להוספה
    hebrew_alphabet = list("אבגדהוזחטיכלמנסעפצקרשתךםןףץ")
    # ניקוד - שימוש בקידוד Unicode ישיר כדי למנוע "קפיצות" ויזואליות בסקריפט
    hebrew_nikud = [
        '\u05b0', '\u05b1', '\u05b2', '\u05b3', '\u05b4', '\u05b5', 
        '\u05b6', '\u05b7', '\u05b8', '\u05b9', '\u05bb', '\u05bc', 
        '\u05c1', '\u05c2'
    ]
    punctuation = [".", ",", "?", "!", ":", ";", "-", '"', " "]

    new_tokens = hebrew_alphabet + hebrew_nikud + punctuation

    # 3. כתיבה לקובץ החדש
    with open(output_file, 'w', encoding='utf-8') as f:
        # קודם כל מעתיקים את המקור
        for line in lines:
            f.write(line + '\n')
        
        # מוסיפים את החדשים עם רווח מובטח
        for token in new_tokens:
            # בדיקה אם התו כבר קיים (כדי לא לייצר כפילויות)
            existing_chars = [l.split(maxsplit=1)[1] for l in lines if len(l.split()) > 1]
            if token in existing_chars:
                continue
                
            # כתיבה פורמלית: {מספר} {רווח} {תו}
            f.write(f"{next_idx} {token}\n")
            next_idx += 1

    print(f"הקובץ סודר! סה\"כ טוקנים: {next_idx}")
    print(f"אינדקס אחרון: {next_idx - 1}")

# הרצה על הקובץ שלך
fix_and_append_vocalized_vocab("voicecraft_data_merged/vocab.txt", "voicecraft_data_merged/vocab_new.txt")

הקובץ סודר! סה"כ טוקנים: 147
אינדקס אחרון: 146


In [11]:
def create_perfect_vocab(input_file, output_file):
    # 1. קריאת 104 הפונמות והסימנים שכבר קיימים (0 עד 103)
    with open(input_file, 'r', encoding='utf-8') as f:
        original_lines = [line.strip() for line in f.readlines() if line.strip()]
    
    # 2. הרשימה המדויקת של מה שבאמת חסר לנו
    hebrew_letters = list("אבגדהוזחטיכלמנסעפצקרשתךםןףץ") # 27 אותיות
    hebrew_nikud = [
        '\u05b0', '\u05b1', '\u05b2', '\u05b3', '\u05b4', '\u05b5', 
        '\u05b6', '\u05b7', '\u05b8', '\u05b9', '\u05bb', '\u05bc', 
        '\u05c1', '\u05c2'
    ] # 14 סימני ניקוד

    only_missing_chars = hebrew_letters + hebrew_nikud
    
    # 3. יצירת הקובץ החדש
    with open(output_file, 'w', encoding='utf-8') as f:
        # קודם כותבים את המקור כמו שהוא
        for line in original_lines:
            f.write(line + '\n')
            
        # עכשיו ממשיכים את המספור מ-104 והלאה
        current_idx = 104
        for token in only_missing_chars:
            f.write(f"{current_idx} {token}\n")
            current_idx += 1

    print(f"הקובץ מוכן! האינדקס האחרון הוא: {current_idx - 1}")
    print(f"ב-YAML (n_text_tokens) צריך לכתוב בדיוק: {current_idx}")

# הרצה על הקובץ שלך
create_perfect_vocab("voicecraft_data_merged/vocab.txt", "voicecraft_data_merged/vocab_new.txt")

הקובץ מוכן! האינדקס האחרון הוא: 144
ב-YAML (n_text_tokens) צריך לכתוב בדיוק: 145
